# Feature Diagnostics & Stationarity Analysis
## Bayesian Regime Detection Engine (Indian Equity Markets)
**Institution:** Zetheta Algorithms Private Limited  
**Corporate Identification Number (CIN):** `U62012MH2023PTC410415`  
**Objective:** Empirical validation of stationarity, correlation structure, and crisis response around 2008, 2013, 2020, and 2022.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.tsa.stattools import adfuller

feat_path = Path("../data/processed/features.parquet")
if not feat_path.exists():
    feat_path = Path("data/processed/features.parquet")

df = pd.read_parquet(feat_path)
print(f"Loaded feature dataset: {df.shape[0]} trading sessions x {df.shape[1]} features.")
print("Date Range:", df.index[0].date(), "to", df.index[-1].date())

In [ ]:
# 1. Augmented Dickey-Fuller (ADF) Stationarity Audit
test_features = ["ret_21d", "vol_21d", "vix_z", "drawdown", "adv_dec_ratio", "usdinr_change_21d", "fii_flow_5d"]
adf_results = []
for f in test_features:
    series = df[f].dropna()
    res = adfuller(series, autolag="AIC")
    adf_results.append({
        "Feature": f,
        "ADF Statistic": round(res[0], 4),
        "p-value": round(res[1], 6),
        "Stationary (p < 0.01)": bool(res[1] < 0.01)
    })
adf_df = pd.DataFrame(adf_results)
print("=== ADF Stationarity Test Results ===")
print(adf_df.to_string(index=False))

In [ ]:
# 2. Cross-Channel Correlation Analysis
core_channels = ["ret_21d", "vol_21d", "vix_level", "drawdown", "breadth_small_minus_large", "fii_flow_5d", "usdinr_change_21d"]
corr_matrix = df[core_channels].corr()
print("=== Cross-Channel Correlation Matrix ===")
print(corr_matrix.round(3))

In [ ]:
# 3. Crisis Episode Diagnostics
# 2013 Taper Tantrum
tt_slice = df.loc["2013-05-01":"2013-09-30"]
max_inr = tt_slice["usdinr_change_21d"].max() * 100
max_gilt = tt_slice["gilt_level"].max()
print(f"2013 Taper Tantrum: Peak INR 21d depreciation: {max_inr:.2f}%, Max Gilt yield: {max_gilt:.2f}%")

# 2020 COVID Shock
covid_slice = df.loc["2020-02-01":"2020-04-30"]
peak_vix = covid_slice["vix_level"].max()
max_dd = covid_slice["drawdown"].min() * 100
peak_alerts = covid_slice["crash_alert_count"].max()
print(f"2020 COVID Crash: Peak VIX: {peak_vix:.1f}, Max Drawdown: {max_dd:.2f}%, Peak Crash Alerts: {peak_alerts}")

# 2022 Inflation & Rate Hike Regime
inf_slice = df.loc["2022-01-01":"2022-06-30"]
max_crude = inf_slice["crude_change_21d"].max() * 100
max_inf_gilt = inf_slice["gilt_level"].max()
print(f"2022 Inflation Shock: Max Crude 21d return: {max_crude:.2f}%, Max Gilt yield: {max_inf_gilt:.2f}%")